In [1]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

# Define input and output folders and files
input_path = '../../Spatial_data_repository'
output_file = '../output/tables/leave_one_country_out_vs_TPS.csv'
model_result_file = '../data/processed/leave_one_country_out_vs_TPS.pkl'

# Prepare data: load lsms data and stacked (raster of drivers)
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')
stacked = rasterio.open('../data/processed/stacked_rasters_africa.tif')
fourteen_countries = ['Benin', 'Burkina', 'Cote_d_Ivoire', 'Ethiopia', 'Guinea_Bissau', 'Malawi', 'Mali', 'Niger', 'Nigeria', 'Senegal', 'Tanzania', 'Togo', 'Uganda', 'Zambia']
fourteen_country_codes = ['BEN', 'BFA', 'CIV', 'ETH', 'GNB', 'MWI', 'MLI', 'NER', 'NGA', 'SEN', 'TZA', 'TGO', 'UGA', 'ZMB']

# Using a training set (all other countries) and a test set (country of interest) to evaluate model performance
def leave_one_country_models(my_country):
    np.random.seed(2024)  # just for reproducibility!
    
    print(f'--------------- Thin plate spline in {my_country} -------------')
    # rename dataset (just to backup original lsms_spatial)
    my_lsms_cty = lsms_spatial.copy()
    
    # training - test split (leave one country out)
    training_set_xy = my_lsms_cty[my_lsms_cty['country'] != my_country].drop(columns=['country']).dropna()
    test_set_xy = my_lsms_cty[my_lsms_cty['country'] == my_country].drop(columns=['country']).dropna()
    
    ## Standardize the data
    #scaler = StandardScaler()
    #X_train_scaled = scaler.fit_transform(training_set_xy.drop(columns=['farm_area_ha']))
    #X_test_scaled = scaler.transform(test_set_xy.drop(columns=['farm_area_ha']))
    X_train_transformed = training_set_xy.drop(columns = ['farm_area_ha'])
    X_test_transformed = test_set_xy.drop(columns = ['farm_area_ha'])
    
    # Random forest with my_country (only the covariates). This serves as reference
    rf_country_ref = RandomForestRegressor(n_estimators=1500, max_features=3, min_samples_leaf=50, random_state=2024)
    rf_country_ref.fit(X_test_transformed, test_set_xy['farm_area_ha'])
    test_set_xy['pred_reference'] = rf_country_ref.predict(X_test_transformed)
    rsq_reference = np.corrcoef(test_set_xy['farm_area_ha'], test_set_xy['pred_reference'])[0, 1] ** 2
    print(f'rsq_reference = {rsq_reference:.2f}')
    
    # Random forest with other countries (only the covariates)
    rf_country_model = RandomForestRegressor(n_estimators=1500, max_features=3, min_samples_leaf=50, random_state=2024)
    rf_country_model.fit(X_train_transformed, training_set_xy['farm_area_ha'])
    test_set_xy['pred_rf'] = rf_country_model.predict(X_test_transformed)
    rsq_rf = np.corrcoef(test_set_xy['farm_area_ha'], test_set_xy['pred_rf'])[0, 1] ** 2
    print(f'rsq_rf = {rsq_rf:.2f}')
    
    # calculate Rsquare showing agreement between in-country TPS and RF_model from other countries
    cor_rf_vs_tps = np.corrcoef(test_set_xy['pred_reference'], test_set_xy['pred_rf'])[0, 1]
    rsq_rf_vs_tps = cor_rf_vs_tps ** 2
    print(f'rsq_rf_vs_tps = {rsq_rf_vs_tps:.2f}')
    
    # compile results
    one_rsq = pd.DataFrame({
        'country': [my_country],
        'rsq_reference': [rsq_reference],
        'rsq_rf': [rsq_rf],
        'rsq_rf_vs_tps': [rsq_rf_vs_tps],
        'cor_rf_vs_tps': [cor_rf_vs_tps]
    })
    
    return one_rsq

# Initialization and function application
start_time = time.time()
results = pd.concat([leave_one_country_models(country) for country in fourteen_countries])
end_time = time.time() - start_time
print(f'Time taken: {end_time:.2f} seconds')

# Save results
results.to_csv(output_file, index=False)

--------------- Thin plate spline in Benin -------------
rsq_reference = 0.42
rsq_rf = 0.28
rsq_rf_vs_tps = 0.74
--------------- Thin plate spline in Burkina -------------
rsq_reference = 0.34
rsq_rf = 0.14
rsq_rf_vs_tps = 0.56
--------------- Thin plate spline in Cote_d_Ivoire -------------
rsq_reference = 0.44
rsq_rf = 0.22
rsq_rf_vs_tps = 0.55
--------------- Thin plate spline in Ethiopia -------------
rsq_reference = 0.31
rsq_rf = 0.11
rsq_rf_vs_tps = 0.33
--------------- Thin plate spline in Guinea_Bissau -------------
rsq_reference = 0.17
rsq_rf = 0.01
rsq_rf_vs_tps = 0.09
--------------- Thin plate spline in Malawi -------------
rsq_reference = 0.25
rsq_rf = 0.06
rsq_rf_vs_tps = 0.26
--------------- Thin plate spline in Mali -------------
rsq_reference = 0.39
rsq_rf = 0.15
rsq_rf_vs_tps = 0.49
--------------- Thin plate spline in Niger -------------
rsq_reference = 0.20
rsq_rf = 0.00
rsq_rf_vs_tps = 0.00
--------------- Thin plate spline in Nigeria -------------
rsq_reference = 